#Xử lí dữ liệu

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# imports
import dask.dataframe as dd
import numpy as np
import pandas as pd
import re

##Xử lí reviews data

In [5]:
#Xem nhanh các cột
df_sample = pd.read_json(
    "/content/drive/MyDrive/Kpdl1/reviews_Electronics.jsonl",
    lines=True,
    nrows=2000
)

df_sample.columns

Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='object')

###Convert file reviewsdata sang parquet và lọc các cột cần thiết

In [ ]:
# reviews
reviews_in = "/content/drive/MyDrive/Kpdl1/reviews_Electronics.jsonl"
reviews_out = "/content/drive/MyDrive/Kpdl1/reviews_Electronics.parquet"

reviews = dd.read_json(reviews_in, lines=True, blocksize="128MB")
reviews_cols = ["parent_asin","user_id","rating","text","timestamp","helpful_vote","verified_purchase"]
reviews = reviews[reviews_cols]
reviews.to_parquet(reviews_out, write_index=False)

###Load reviewdata

In [3]:
reviews = dd.read_parquet("/content/drive/MyDrive/Kpdl1/reviews_Electronics.parquet")
print("Số dòng:", reviews.shape[0].compute())
print("Cột:", list(reviews.columns))
reviews.head()

Số dòng: 43886944
Cột: ['parent_asin', 'user_id', 'rating', 'text', 'timestamp', 'helpful_vote', 'verified_purchase']


,parent_asin,user_id,rating,text,timestamp,helpful_vote,verified_purchase
0,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,3,First & most offensive: they reek of gasoline ...,2022-07-18 22:58:37.948,0,True
1,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1,These didn’t work. Idk if they were damaged in...,2020-06-20 18:42:29.731,0,True
2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,5,I love these. They even come with a carry case...,2018-04-07 09:23:37.534,0,True
3,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,5,I was searching for a sturdy backpack for scho...,2010-11-20 18:41:35.000,18,True
4,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,5,I've bought these headphones three times becau...,2023-02-17 02:39:41.238,0,True


In [7]:
#check missing
missing = reviews.isna().mean().compute().sort_values(ascending=False)
missing

,0
parent_asin,0.0
user_id,0.0
rating,0.0
text,0.0
timestamp,0.0
helpful_vote,0.0
verified_purchase,0.0


In [8]:
parent_empty = (reviews["parent_asin"].astype("string").str.strip() == "").mean().compute()
text_empty = (reviews["text"].astype("string").str.strip() == "").mean().compute()

{"ti_le_parent_asin_rong": float(parent_empty), "ti_le_text_rong": float(text_empty)}

{'ti_le_parent_asin_rong': 0.0, 'ti_le_text_rong': 0.0009386162773147294}

In [4]:
# Tạo độ dài review
reviews["text_length"] = reviews["text"].astype("string").str.strip().str.len().astype("int64")

# Thống kê
stats = reviews["text_length"].describe().compute()
stats = stats.apply(lambda x: f"{x:,.0f}")
stats = stats.to_frame("value")
stats

,value
count,"43,886,944"
mean,241
std,419
min,0
25%,54
50%,140
75%,326
max,"35,208"


###Lọc review không có ý nghĩa (dưới 20 kí tự)

In [5]:
# Số review trước khi lọc
before_count = reviews.shape[0].compute()

# Lọc review rỗng và quá ngắn
reviews = reviews[reviews["text_length"] >= 20]

# Số review sau khi lọc
after_count = reviews.shape[0].compute()

print("Trước lọc:", before_count)
print("Sau lọc:", after_count)
print("Đã xóa:", before_count - after_count)

Trước lọc: 43886944
Sau lọc: 39323727
Đã xóa: 4563217


In [6]:
# Thống kê phân bố rating + lọc rating hợp lệ (1–5)

print(reviews["rating"].value_counts().compute().sort_index())

before = int(reviews.shape[0].compute())
reviews = reviews[reviews["rating"].between(1, 5)]
after = int(reviews.shape[0].compute())

print(f"Trước lọc: {before:,}")
print(f"Sau lọc: {after:,}")
print(f"Đã xoá: {before-after:,}")

rating
0           2
1     5127621
2     2200406
3     2758529
4     5102940
5    24134229
Name: count, dtype: int64
Trước lọc: 39,323,727
Sau lọc: 39,323,725
Đã xoá: 2


In [7]:
# Làm sạch text + loại text rỗng sau làm sạch
# 1) chuẩn hoá text
reviews["text"] = reviews["text"].astype("string").str.lower()
reviews["text"] = reviews["text"].str.replace(r"[^a-z0-9\s]", "", regex=True)
reviews["text"] = reviews["text"].str.strip()

# 2) loại review có text rỗng sau làm sạch
before = int(reviews.shape[0].compute())
reviews = reviews[reviews["text"].notnull() & (reviews["text"].str.len() > 0)]
after = int(reviews.shape[0].compute())

print(f"Trước lọc text rỗng: {before:,}")
print(f"Sau lọc text rỗng: {after:,}")
print(f"Đã xoá: {before-after:,}")


Trước lọc text rỗng: 39,323,725
Sau lọc text rỗng: 39,323,435
Đã xoá: 290


###Lưu file

In [10]:
# Lưu dữ liệu review đã làm sạch (đúng thư mục Kpdl1)
save_path = "/content/drive/MyDrive/Kpdl1/clean_reviews.parquet"
reviews.to_parquet(save_path, write_index=False)

##Xử lí metadata

###Load metadata

In [4]:
import dask.dataframe as dd

path = "/content/drive/MyDrive/Kpdl1/meta_Electronics/raw_meta_Electronics/*.parquet"

df_meta = dd.read_parquet(path)

print("Số dòng:", df_meta.shape[0].compute())
print("Số cột:", df_meta.shape[1])

df_meta.info()
df_meta.head()

Số dòng: 1610012
Số cột: 16
<class 'dask.dataframe.dask_expr.DataFrame'>
Columns: 16 entries, main_category to author
dtypes: object(5), float64(1), int64(1), string(9)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Fat Shark,"[Electronics, Television & Video, Video Glasses]","{""Date First Available"": ""August 2, 2014"", ""Ma...",B00MCW7G9M,<NA>,<NA>,<NA>
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SIIG,"[Electronics, Television & Video, Accessories,...","{""Product Dimensions"": ""0.83 x 4.17 x 2.05 inc...",B00YT6XQSE,<NA>,<NA>,<NA>
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['AL 2Sides Video', 'MacBook Protect...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{""Brand"": ""Digi-Tatoo"", ""Color"": ""Fresh Marble...",B07SM135LS,<NA>,<NA>,<NA>
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{""Date First Available"": ""May 29, 2020"", ""Manu...",B089CNGZCW,<NA>,<NA>,<NA>
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",Verizon,"[Electronics, Computers & Accessories, Compute...","{""Product Dimensions"": ""11.6 x 6.9 x 3.1 inche...",B004E2Z88O,<NA>,<NA>,<NA>





###Giữ lại các cột cần thiết

In [14]:
cols_needed = [
    'parent_asin',
    'title',
    'main_category',
    'price',
    'average_rating',
    'rating_number',
]

df_meta = df_meta[cols_needed]
df_meta.head()



,parent_asin,title,main_category,price,average_rating,rating_number
0,B00MCW7G9M,FS-1051 FATSHARK TELEPORTER V3 HEADSET,All Electronics,None,3.5,6
1,B00YT6XQSE,Ce-H22B12-S1 4Kx2K Hdmi 4Port,All Electronics,None,5.0,1
2,B07SM135LS,Digi-Tatoo Decal Skin Compatible With MacBook ...,Computers,19.99,4.5,246
3,B089CNGZCW,NotoCity Compatible with Vivoactive 4 band 22m...,AMAZON FASHION,9.99,4.5,233
4,B004E2Z88O,Motorola Droid X Essentials Combo Pack,Cell Phones & Accessories,14.99,3.8,64


###Xử lý giá trị thiếu

In [15]:
# Kiểm tra số lượng thiếu
df_meta.isnull().sum().compute().to_frame(name="missing_count")

,missing_count
parent_asin,0
title,0
main_category,106334
price,0
average_rating,0
rating_number,0


In [16]:
#Xử lí giá trị thiếu
df_meta['main_category'] = df_meta['main_category'].fillna('Unknown')

df_meta.isnull().sum().compute().to_frame(name="missing_count")

,missing_count
parent_asin,0
title,0
main_category,0
price,0
average_rating,0
rating_number,0


###Xử lí cột price

In [17]:
# Hàm convert price về float
def parse_price(value):
    if value is None:
        return np.nan

    # nếu đã là số thì kiểm tra luôn
    if isinstance(value, (int, float, np.integer, np.floating)):
        v = float(value)
        return v if np.isfinite(v) and v > 0 else np.nan

    text = str(value).strip().lower()

    # bỏ các giá trị rỗng hoặc free
    if text in {"", "none", "nan", "null", "na", "<na>"} or "free" in text:
        return np.nan

    # bỏ ký hiệu tiền tệ và dấu phẩy
    text = re.sub(r"[,$]", "", text)
    text = text.replace("usd", "").replace("us$", "").replace("$", "")

    # xử lý dạng khoảng giá 10-20 hoặc 10 to 20
    m = re.search(r"(\d+(?:\.\d+)?)\s*(?:-|to)\s*(\d+(?:\.\d+)?)", text)
    if m:
        a = float(m.group(1))
        b = float(m.group(2))
        v = (a + b) / 2
        return v if v > 0 else np.nan

    # lấy số lớn nhất nếu có nhiều số trong chuỗi
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    if not nums:
        return np.nan

    v = max(float(x) for x in nums)
    return v if v > 0 else np.nan


# áp dụng hàm parse
df_meta["price_clean"] = df_meta["price"].apply(
    parse_price,
    meta=("price_clean", "float32")
)

# ép chắc chắn về numeric
df_meta["price_clean"] = dd.to_numeric(
    df_meta["price_clean"],
    errors="coerce"
).astype("float32")

# loại các giá trị <= 0
df_meta["price_clean"] = df_meta["price_clean"].mask(
    df_meta["price_clean"] <= 0,
    np.nan
)

# xem thử kết quả
df_meta[["title", "price", "price_clean"]].head(5)

,title,price,price_clean
0,FS-1051 FATSHARK TELEPORTER V3 HEADSET,None,NaN
1,Ce-H22B12-S1 4Kx2K Hdmi 4Port,None,NaN
2,Digi-Tatoo Decal Skin Compatible With MacBook ...,19.99,19.99
3,NotoCity Compatible with Vivoactive 4 band 22m...,9.99,9.99
4,Motorola Droid X Essentials Combo Pack,14.99,14.99


In [18]:
# lọc giá trị lỗi cho 2 cột rating và avg rating
df_meta["average_rating"] = df_meta["average_rating"].where(df_meta["average_rating"].between(1,5), np.nan)
df_meta["rating_number"]  = df_meta["rating_number"].where(df_meta["rating_number"] >= 0, np.nan)

# bỏ cột price
df_meta = df_meta.drop(columns=["price"], errors="ignore")

In [19]:
# Kiểm tra sản phẩm trùng theo parent_asin
total = int(df_meta.shape[0].compute())
unique = int(df_meta["parent_asin"].nunique().compute())
print("Số dòng bị trùng:", total - unique)

# Xoá trùng, giữ lại dòng đầu tiên
df_meta = df_meta.drop_duplicates(subset=["parent_asin"])
print("Số dòng sau khi xoá trùng:", int(df_meta.shape[0].compute()))

Số dòng bị trùng: 0
Số dòng sau khi xoá trùng: 1610012


In [20]:
# Làm sạch title: bỏ khoảng trắng thừa và loại sản phẩm không có tên
df_meta["title"] = df_meta["title"].fillna("").str.strip()
df_meta = df_meta[df_meta["title"] != ""]
print("Số dòng còn lại:", int(df_meta.shape[0].compute()))

Số dòng còn lại: 1609918


In [23]:
# Lưu metadata clean
save_path = "/content/drive/MyDrive/Kpdl1/clean_metadata.parquet"

df_meta.to_parquet(save_path, write_index=False)

##Merge data